# PG Data Service — Card-Based API Validation

Validate `get_card()` output against the Domo Ad Performance card CSV export.

Tests both output modes:
- **summary**: `get_card("ad_performance_summary", ...)` — one row per ad, snake_case columns
- **daily**: `get_card("ad_performance_daily", ...)` — one row per ad per day, Domo display names

In [22]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 200)

## 1. Summary Card (snake_case columns, one row per ad)

In [ ]:
from api import get_card, list_cards

DATE_FROM = "2026-03-08"
DATE_TO = "2026-03-14"

print("Available cards:", list_cards())
print()

summary = get_card("ad_performance_summary", DATE_FROM, DATE_TO)
print(f"{len(summary)} ads, {len(summary.columns)} columns")
summary.head(3)

In [ ]:
# All available columns in summary output
print("Summary columns:")
for c in summary.columns:
    print(f"  {c}")

## 2. Daily Card (Domo display names, one row per ad per day)

In [ ]:
daily = get_card("ad_performance_daily", DATE_FROM, DATE_TO)
print(f"{len(daily)} rows, {daily['Ad'].nunique()} ads, {daily['Day'].nunique()} days, {len(daily.columns)} columns")
daily.head(5)

In [ ]:
# Top 10 by spend (aggregate across days for ranking)
daily_spend = daily.groupby("Ad")["Spend"].sum().nlargest(10).index
daily[daily["Ad"].isin(daily_spend)][["Ad", "Day", "Spend", "Net Revenue", "Net ROAS", "Gross Revenue", "Orders", "# SC Trials<BR>Started", "NC Orders"]]

## 3. Compare Daily Card Against Domo CSV

Export a fresh CSV from the Domo Ad Performance card for the same date range (3/8–3/14) and save it to `notebooks/domo_ad_performance.csv`.

In [ ]:
DOMO_CSV = "domo_ad_performance.csv"

domo = pd.read_csv(DOMO_CSV)
domo["Ad"] = domo["Ad"].astype(str).str.lower().str.strip()
print(f"Domo CSV: {len(domo)} rows, {domo['Day'].nunique()} days ({sorted(domo['Day'].unique())}), {domo['Ad'].nunique()} unique ads")

In [ ]:
# Aggregate both to per-ad totals for comparison
additive = ["Spend", "Gross Revenue", "Clicks", "Impressions", "Orders",
            "# SC Trials<BR>Started", "NC Orders", "Net Revenue", "Fixed Refund<BR>Net Revenue"]

for c in additive:
    if c in domo.columns:
        domo[c] = pd.to_numeric(domo[c], errors="coerce").fillna(0)

domo_agg = domo.groupby("Ad", as_index=False)[additive].sum()

# Aggregate our daily card rows too
ours = daily.copy()
ours["Ad"] = ours["Ad"].astype(str).str.lower().str.strip()
for c in additive:
    if c in ours.columns:
        ours[c] = pd.to_numeric(ours[c], errors="coerce").fillna(0)
ours_agg = ours.groupby("Ad", as_index=False)[additive].sum()

print(f"Domo: {len(domo_agg)} ads")
print(f"Ours: {len(ours_agg)} ads")

In [29]:
# Find common ads
common = set(domo_agg["Ad"]) & set(ours_agg["Ad"])
print(f"Common ads: {len(common)}")
print(f"Domo-only: {len(set(domo_agg['Ad']) - set(ours_agg['Ad']))}")
print(f"Ours-only: {len(set(ours_agg['Ad']) - set(domo_agg['Ad']))}")

Common ads: 928
Domo-only: 0
Ours-only: 17


In [30]:
# Side-by-side: top 10 by spend
d = domo_agg[domo_agg["Ad"].isin(common)].set_index("Ad")
o = ours_agg[ours_agg["Ad"].isin(common)].set_index("Ad")

top_ads = d.nlargest(10, "Spend").index

for col in additive:
    domo_vals = d.loc[top_ads, col]
    ours_vals = pd.to_numeric(o.reindex(top_ads)[col], errors="coerce")
    diff = (ours_vals - domo_vals).abs()
    rel = diff / domo_vals.abs().clip(lower=0.01)
    status = "OK" if rel.max() <= 0.01 else "MISMATCH"
    print(f"  {status:10s} | {col:35s} | max diff: {rel.max():.4%}")

  OK         | Spend                               | max diff: 0.0000%
  OK         | Gross Revenue                       | max diff: 0.0000%
  OK         | Clicks                              | max diff: 0.0000%
  OK         | Impressions                         | max diff: 0.1332%
  OK         | Orders                              | max diff: 0.0000%
  OK         | # SC Trials<BR>Started              | max diff: 0.0000%
  OK         | NC Orders                           | max diff: 0.0000%
  OK         | Net Revenue                         | max diff: 0.0000%
  OK         | Fixed Refund<BR>Net Revenue         | max diff: 0.0000%


In [31]:
# Detailed side-by-side for top 5 ads
for ad in top_ads[:5]:
    print(f"\n{'='*80}")
    print(f"Ad: {ad[:70]}")
    print(f"{'='*80}")
    for col in additive:
        d_val = d.loc[ad, col]
        o_val = pd.to_numeric(o.loc[ad, col], errors="coerce") if ad in o.index else float("nan")
        match = "ok" if abs(d_val - o_val) / max(abs(d_val), 0.01) < 0.01 else "!!"
        print(f"  {col:35s}  domo={d_val:>15.2f}  ours={o_val:>15.2f}  {match}")


Ad: pmax
  Spend                                domo=       37241.99  ours=       37241.99  ok
  Gross Revenue                        domo=       91319.50  ours=       91319.50  ok
  Clicks                               domo=       28537.00  ours=       28537.00  ok
  Impressions                          domo=     3137062.00  ours=     3137062.00  ok
  Orders                               domo=         299.00  ours=         299.00  ok
  # SC Trials<BR>Started               domo=         275.00  ours=         275.00  ok
  NC Orders                            domo=         222.00  ours=         222.00  ok
  Net Revenue                          domo=       20279.49  ours=       20279.49  ok
  Fixed Refund<BR>Net Revenue          domo=       16657.93  ours=       16657.93  ok

Ad: shopping ad
  Spend                                domo=       17026.25  ours=       17026.25  ok
  Gross Revenue                        domo=       37208.00  ours=       37208.00  ok
  Clicks                   

## 4. Quick Tess Classification Preview (using summary card)

In [ ]:
# Classification using Tess PRD v1.4 thresholds (on summary card)
# def classify(row):
#     if row["spend"] < 2500:
#         return "Testing"
#     elif row["net_roas"] >= 1.0:
#         return "Winner"
#     elif row["net_roas"] >= 0.80:
#         return "Potential"
#     else:
#         return "Underperformer"

# summary["classification"] = summary.apply(classify, axis=1)
# print(summary["classification"].value_counts())
# print(f"\nTotal spend: ${summary['spend'].sum():,.2f}")
# print(f"Total revenue: ${summary['gross_revenue'].sum():,.2f}")

In [ ]:
# # Winners
# winners = summary[summary["classification"] == "Winner"].nlargest(10, "spend")
# winners[["ad_name", "funnel", "spend", "gross_revenue", "net_roas", "nc_net_roas", "cpa", "sc_trials"]]

In [ ]:
# # Underperformers (spend >= $2500 but net_roas < 0.80)
# under = summary[summary["classification"] == "Underperformer"].nlargest(10, "spend")
# under[["ad_name", "funnel", "spend", "gross_revenue", "net_roas", "nc_net_roas", "cpa", "sc_trials"]]

In [ ]:
# # Breakdown by funnel
# by_funnel = summary.groupby("funnel").agg(
#     ads=("ad_name", "count"),
#     total_spend=("spend", "sum"),
#     total_revenue=("gross_revenue", "sum"),
#     winners=("classification", lambda x: (x == "Winner").sum()),
# ).sort_values("total_spend", ascending=False)

# by_funnel["roas"] = by_funnel["total_revenue"] / by_funnel["total_spend"]
# by_funnel